# GenAI as a Force Multiplier in Cloud Security
### M.Tech Dissertation — Tushar Sood, JIIT 2026
**Run this notebook top-to-bottom. Each section corresponds to one architecture layer.**

**Free setup:**
- Runtime → Change runtime type → **T4 GPU**
- For LLM (Layer 3): Get free Groq API key at https://console.groq.com
- Everything else is completely free

In [ ]:
# ── CELL 1: Mount Drive + Clone Repo ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive')

# Clone your repo (replace with your GitHub URL after uploading)
# !git clone https://github.com/YOUR_USERNAME/genai_cloud_security

# OR upload the zip and extract:
# from google.colab import files
# files.upload()  # Upload genai_cloud_security.zip
# !unzip genai_cloud_security.zip

os.chdir('/content/drive/MyDrive/genai_cloud_security')
!ls

In [ ]:
# ── CELL 2: Install Dependencies ─────────────────────────────────────────────
!pip install -q torch torchvision scikit-learn pandas numpy tqdm mlflow
!pip install -q sentence-transformers faiss-cpu shap
!pip install -q langchain langchain-community
!pip install -q groq  # Free LLM API
print('All dependencies installed!')

In [ ]:
# ── CELL 3: Set API Keys (Groq is FREE) ──────────────────────────────────────
import os

# Get your FREE key at: https://console.groq.com/keys
os.environ['GROQ_API_KEY'] = 'your_groq_key_here'
os.environ['LLM_PROVIDER'] = 'groq'

# If you want fully offline (no API key needed):
# os.environ['LLM_PROVIDER'] = 'offline'

print('Config set. Provider:', os.environ.get('LLM_PROVIDER'))

In [ ]:
# ── CELL 4: Download Datasets ────────────────────────────────────────────────
!python utils/download_datasets.py

## Layer 1: Synthetic Data Generation (DDPM + GAN)
Expected result: DDPM-augmented IDS outperforms baseline on minority attack classes.

In [ ]:
# ── CELL 5: Layer 1 ───────────────────────────────────────────────────────────
# Tip: Takes ~15-20 min on Colab free T4. Reduce epochs in config.py for quick test.
# config.L1['ddpm_epochs'] = 5  # Quick mode

from layer1_data.train_ddpm import run_layer1_experiment
l1_results = run_layer1_experiment()

## Layer 2: Intrusion Detection + Adversarial Purification
Expected: 99%+ accuracy with Attention-IDS. Near-zero under FGSM attack. Restored to 90%+ after purification.

In [ ]:
# ── CELL 6: Layer 2 ───────────────────────────────────────────────────────────
from layer2_detection.train_ids import run_layer2_experiment
l2_results = run_layer2_experiment()

## Layer 3: LLM + RAG Cognitive Analysis
Expected: ~80% MITRE ATT&CK mapping accuracy. RAG improves over zero-shot.

In [ ]:
# ── CELL 7: Layer 3 ───────────────────────────────────────────────────────────
from layer3_cognitive.run_rag_pipeline import run_layer3_experiment
l3_results = run_layer3_experiment()

## Layer 4: Autonomous Response Simulation
Expected: 99%+ MTTR reduction vs 8-hour manual baseline.

In [ ]:
# ── CELL 8: Layer 4 ───────────────────────────────────────────────────────────
from layer4_response.autonomous_response import run_layer4_experiment
l4_results = run_layer4_experiment()

## Layer 5: Governance + Results Dashboard
Generates your main dissertation figure.

In [ ]:
# ── CELL 9: Layer 5 + Dashboard ──────────────────────────────────────────────
from layer5_governance.governance import run_layer5_experiment
l5_results = run_layer5_experiment()

In [ ]:
# ── CELL 10: View Dashboard ──────────────────────────────────────────────────
from IPython.display import Image
Image('/content/drive/MyDrive/genai_cloud_security/results/DISSERTATION_RESULTS_DASHBOARD.png')

In [ ]:
# ── CELL 11: MLflow Results ──────────────────────────────────────────────────
# View all experiment metrics
import mlflow
import sys
sys.path.insert(0, '.')
from config import MLFLOW_TRACKING_URI

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name('genai_cloud_security_dissertation')
if exp:
    runs = client.search_runs(exp.experiment_id)
    for run in runs:
        print(f"Run: {run.info.run_name}")
        for k, v in run.data.metrics.items():
            print(f"  {k}: {v:.4f}")
        print()

In [ ]:
# ── CELL 12: Download all results ───────────────────────────────────────────
import shutil
shutil.make_archive('/content/dissertation_results', 'zip',
                    '/content/drive/MyDrive/genai_cloud_security/results')
from google.colab import files
files.download('/content/dissertation_results.zip')